In [ ]:
import scanpy as sc
from tqdm import tqdm
import numpy as np
import pandas as pd
import seaborn as sns

Read AnnData 

In [ ]:
adata = sc.read_h5ad("../../data_dir/replogle.h5ad")
adata = adata[:, adata.var["highly_variable"]]

## HepG2 test holdout (official STATE split)

`split2` follows [ArcInstitute/state `hepg2.toml`](https://github.com/ArcInstitute/state/blob/main/state_preprint_toml_files/replogle_tomls/hepg2.toml): **946 test perturbations** on HepG2 (including `non-targeting`); all other HepG2 genes → `finetune`. No separate `val` phase.

In [ ]:
from pathlib import Path

REPO = Path("../../")
HEPG2_TEST_GENES_PATH = REPO / "configs/replogle_hepg2_state_test_genes.txt"
# Source: ArcInstitute/state state_preprint_toml_files/replogle_tomls/hepg2.toml
# (946 genes incl. non-targeting; no val split)

hepg2_test_genes = HEPG2_TEST_GENES_PATH.read_text().splitlines()
hepg2_test_genes = [g for g in hepg2_test_genes if g and not g.startswith("#")]
hepg2_test_genes = sorted(set(hepg2_test_genes))
print(f"Loaded {len(hepg2_test_genes)} official HepG2 test genes from {HEPG2_TEST_GENES_PATH}")

hepg2_mask = adata.obs["cell_line"] == "hepg2"
hepg2_genes_in_adata = set(adata.obs.loc[hepg2_mask, "gene"].astype(str))
missing = sorted(set(hepg2_test_genes) - hepg2_genes_in_adata)
if missing:
    raise ValueError(
        f"{len(missing)} official test genes missing from adata on HepG2, e.g. {missing[:5]}"
    )
extra_test = sorted(hepg2_genes_in_adata - set(hepg2_test_genes))
print(
    f"HepG2: {len(hepg2_genes_in_adata)} profiled perturbations; "
    f"{len(hepg2_test_genes)} official test; "
    f"{len(extra_test)} finetune/support"
)


In [ ]:
print(adata.X.max(1)) 

# Perform splits 

## Split 1: train / heldout 

In [ ]:
adata.obs["split1"] = "train"
adata.obs.loc[adata.obs["cell_line"] == "hepg2", "split1"] = "finetune"
print(adata.obs["split1"].value_counts())

## Split 2: train / finetune / test (official STATE HepG2)

- Non-HepG2 cell lines → `train` (pretrain)
- HepG2 genes **not** in the official test list → `finetune` (~30% of HepG2 perts)
- HepG2 genes in `configs/replogle_hepg2_state_test_genes.txt` → `test` (~70%, eval only)
- Exception: `non-targeting` is in the toml test list but stays in `finetune`, so HepG2 has its own controls. `test` never holds controls; they are injected at eval time.


In [ ]:
adata.obs["split2"] = "train"
adata.obs.loc[adata.obs["cell_line"] == "hepg2", "split2"] = "finetune"
# non-targeting stays trainable: every train/finetune context needs its own controls.
adata.obs.loc[
    (adata.obs["cell_line"] == "hepg2")
    & (adata.obs["gene"].isin(hepg2_test_genes))
    & (adata.obs["gene"] != "non-targeting"),
    "split2",
] = "test"
print(adata.obs["split2"].value_counts())
h = adata.obs[adata.obs["cell_line"] == "hepg2"]
print(
    f"HepG2 split2: finetune={h['split2'].eq('finetune').sum():,} cells "
    f"({h.loc[h['split2']=='finetune','gene'].nunique()} perts), "
    f"test={h['split2'].eq('test').sum():,} cells "
    f"({h.loc[h['split2']=='test','gene'].nunique()} perts)"
)

In [ ]:
adata_train_ft = adata[(adata.obs["split2"] == "train") | (adata.obs["split2"] == "finetune")]
sc.tl.pca(adata_train_ft, n_comps=256)

In [ ]:
adata.varm["PCs"] = adata_train_ft.varm["PCs"]
adata.uns["means_train"] = adata_train_ft.X.mean(0)[None, :]
adata.obsm["X_pca"] = (adata.X - adata.uns["means_train"]) @ adata.varm["PCs"]

In [ ]:
sc.pl.pca(adata, color="split2", s=5)

## Neighbors / UMAP on PCA (visualization only)

`X_pca` is already stored. Neighbors/UMAP below are for QC plots; they are not required for flow-matching training.


In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color=["cell_line","split1","split2"], s=5)

In [ ]:
sc.pl.umap(adata, color=["gene"], groups=["non-targeting"], s=5)

In [ ]:
adata.obs = adata.obs.loc[:, ["gene", "sgID_AB", "cell_line", "split1", "split2"]]

## Store PCA and STATE (SE) embeddings on the same AnnData

PCA and SE are **cell embeddings** (256-D vs ~2048-D), so they go in `adata.obsm`, not `adata.layers` (layers must match gene `X`). Training uses `--obsm-key X_pca` or `--obsm-key X_state`.

- **PCA** is already in `obsm["X_pca"]` (fit on `split2` train+finetune; val/test held out of the fit).
- **SE-600M** is copied from the encode dump into `obsm["X_state"]` (frozen encoder; same cell order).

If the dump is missing, encode then re-run this cell:

```bash
SKIP_DOWNLOAD=1 INPUT_H5AD=data_dir/replogle_with_splits.h5ad \
  OUTPUT_H5AD=results_dir/arc_state/se/replogle_with_splits_se.h5ad \
  sbatch sbatch/third_party/encode_se.sbatch
```
(You can encode from `data_dir/replogle.h5ad` **before** this write if cell order is unchanged.)


In [ ]:
SE_PATH = REPO / "results_dir/arc_state/se/replogle_with_splits_se.h5ad"
SE_KEY = "X_state"


def attach_state_embeddings(adata, se_path, *, key=SE_KEY) -> bool:
    """Copy frozen SE-600M vectors into ``adata.obsm[key]`` (aligned by obs_names)."""
    se_path = Path(se_path)
    if not se_path.exists():
        print(
            f"SE dump not found: {se_path}\n"
            "Skipping obsm['X_state']. Encode with sbatch/third_party/encode_se.sbatch "
            "then re-run this cell and the write."
        )
        return False
    if se_path.suffix == ".npy":
        z = np.load(se_path, mmap_mode="r")
        if z.shape[0] != adata.n_obs:
            raise ValueError(
                f"SE npy has {z.shape[0]} rows but adata has {adata.n_obs} cells."
            )
        adata.obsm[key] = np.asarray(z, dtype=np.float32)
        return True
    se = sc.read_h5ad(se_path)
    if key not in se.obsm:
        raise KeyError(f"obsm[{key!r}] missing in {se_path}; have {list(se.obsm)}")
    if not se.obs_names.equals(adata.obs_names):
        missing = adata.obs_names.difference(se.obs_names)
        if len(missing):
            raise ValueError(
                f"{len(missing)} cells in adata missing from SE dump, e.g. {list(missing[:5])}"
            )
        se = se[adata.obs_names].copy()
    adata.obsm[key] = np.asarray(se.obsm[key], dtype=np.float32)
    return True


if "X_pca" not in adata.obsm:
    raise RuntimeError("Run the PCA cells first (obsm['X_pca']).")

attached = attach_state_embeddings(adata, SE_PATH)
adata.uns["pca_obsm_key"] = "X_pca"
adata.uns["se_obsm_key"] = SE_KEY if attached else None
adata.uns["se_source"] = str(SE_PATH) if attached else None
print(f"obsm keys: {list(adata.obsm.keys())}")
if attached:
    print(
        f"X_pca {adata.obsm['X_pca'].shape}  "
        f"X_state {adata.obsm[SE_KEY].shape}  "
        f"finite SE={bool(np.isfinite(adata.obsm[SE_KEY]).all())}"
    )


In [ ]:
out_path = REPO / "data_dir/replogle_with_splits.h5ad"
adata.write_h5ad(out_path)
print(f"Wrote {out_path}")
print("obsm keys:", list(adata.obsm.keys()))


In [ ]:
len(adata.obs.loc[adata.obs["split2"]=="test", "gene"].unique())

## Check PCA distance perturbation controls and compare ranks 

In [ ]:
adata = sc.read_h5ad("../../data_dir/replogle_with_splits.h5ad")

In [ ]:
strata_ctr_distances = {}

for cell_line in tqdm(adata.obs["cell_line"].unique()):
    strata_ctr_distances[cell_line] = {}
    print(cell_line)
    X_cl_ctr = adata.obsm["X_pca"][(adata.obs["cell_line"] == cell_line) & (adata.obs["gene"] == "non-targeting"), :]
    X_cl_ctr_mean = X_cl_ctr.mean(0)
    for pert in adata.obs["gene"].unique():
        if pert == "non-targeting":
            continue
        X_pert = adata.obsm["X_pca"][(adata.obs["cell_line"] == cell_line) & (adata.obs["gene"] == pert), :]
        X_pert_mean = X_pert.mean(0)
        dists = np.linalg.norm(X_cl_ctr_mean - X_pert_mean)
        strata_ctr_distances[cell_line][pert] = [dists]

In [ ]:
cell_line_dfs = {}
for cell_line in adata.obs["cell_line"].unique():
    cell_line_dfs[cell_line] = pd.DataFrame(strata_ctr_distances[cell_line]).T
    cell_line_dfs[cell_line].columns = ["Dist"]
    cell_line_dfs[cell_line] = cell_line_dfs[cell_line][~cell_line_dfs[cell_line]["Dist"].isna()]
    cell_line_dfs[cell_line] = cell_line_dfs[cell_line].sort_values(by="Dist", ascending=False).reset_index()
    cell_line_dfs[cell_line].head()


## Compute rank correlation 

In [ ]:
shared_pool = cell_line_dfs["hepg2"]["index"].tolist()

for cell_line in cell_line_dfs:
    shared_pool = np.intersect1d(shared_pool, cell_line_dfs[cell_line]["index"].tolist())

len(shared_pool)

In [ ]:
for cell_line in cell_line_dfs:
    cell_line_dfs[cell_line] = cell_line_dfs[cell_line].loc[cell_line_dfs[cell_line]["index"].isin(shared_pool)].reset_index(drop=True)
    cell_line_dfs[cell_line]["rank"]  = cell_line_dfs[cell_line].index.tolist()
    print(cell_line_dfs[cell_line].shape)

In [ ]:
rank_corr = pd.DataFrame(
    index=["hepg2", "jurkat", "k562", "rpe1"],
    columns=["hepg2", "jurkat", "k562", "rpe1"],
    dtype=float,
)

for cell_line1 in cell_line_dfs:
    for cell_line2 in cell_line_dfs:
        if cell_line1 == cell_line2:
            rank_corr.loc[cell_line1, cell_line2] = 1.0
            continue
        # Collect gene ranks for both cell type
        ct1_df_tmp = cell_line_dfs[cell_line1].set_index("index")
        ct2_df_tmp = cell_line_dfs[cell_line2].set_index("index")
        # reorder rows of second df to match order of first df
        ct2_df_tmp = ct2_df_tmp.loc[ct1_df_tmp.index]
        # compute rank correlation
        rank_corr.loc[cell_line1, cell_line2] = ct1_df_tmp["rank"].corr(ct2_df_tmp["rank"])

rank_corr
        

In [ ]:
# Moved to last cell (needs float dtype — see rank_corr construction)
rank_corr.astype(float)

In [ ]:
rank_corr


In [ ]:
import matplotlib.pyplot as plt

# rank_corr must be numeric (see previous cell: dtype=float)
rank_corr_f = rank_corr.astype(float)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    rank_corr_f,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=0.5,
    vmax=1.0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Spearman ρ (perturbation ranks)"},
    ax=ax,
)
ax.set_title("Perturbation effect rank correlation across cell lines")
plt.tight_layout()
plt.show()

# Test PCA / SE test set reconstruction 

First assess how test and finetune reconstruction change  in prediction if you don't include them in the training set. 

In [ ]:
adata_train = adata[adata.obs["split2"] == "train"]
adata_finetune = adata[adata.obs["split2"] == "finetune"]
adata_test = adata[adata.obs["split2"] == "test"]

In [ ]:
sc.pp.pca(adata_train)


In [ ]:
PCs_train = adata_train.varm["PCs"]
means_train = adata_train.X.mean(0)[None, :]

# Finetune recon
z_ft = adata_finetune.X - means_train
z_ft = z_ft @ PCs_train
X_recon_ft = z_ft @ PCs_train.T + means_train


# Test set recon 
z_test = adata_test.X - means_train
z_test = z_test @ PCs_train
X_recon_test = z_test @ PCs_train.T + means_train

In [ ]:
adata_test_with_recon = sc.AnnData(X=np.concatenate([adata_test.X.toarray(), X_recon_test], axis=0))
obs_test_with_recon = pd.DataFrame({"dataset_type": ["true"] * adata_test.n_obs + ["recon"] * adata_test.n_obs})
adata_test_with_recon.obs = obs_test_with_recon

In [ ]:
adata_finetune_with_recon = sc.AnnData(X=np.concatenate([adata_finetune.X.toarray(), X_recon_ft], axis=0))
obs_finetune_with_recon = pd.DataFrame({"dataset_type": ["true"] * adata_finetune.n_obs + ["recon"] * adata_finetune.n_obs})
adata_finetune_with_recon.obs = obs_finetune_with_recon

In [ ]:
sc.pp.pca(adata_test_with_recon)
sc.pp.neighbors(adata_test_with_recon)
sc.tl.umap(adata_test_with_recon)
sc.pl.umap(adata_test_with_recon, color=["dataset_type"], s=5)

In [ ]:
sc.pp.pca(adata_finetune_with_recon )
sc.pp.neighbors(adata_finetune_with_recon)
sc.tl.umap(adata_finetune_with_recon)
sc.pl.umap(adata_finetune_with_recon, color=["dataset_type"], s=5)


What if I'd included the ft predictions 

In [ ]:
adata_train_and_ft = adata[adata.obs["split2"].isin(["train", "finetune"])]
sc.tl.pca(adata_train_and_ft)

PCs_train_and_ft = adata_train_and_ft.varm["PCs"]
means_train_and_ft = adata_train_and_ft.X.mean(0)[None, :]

In [ ]:
# Test set recon 
z_test_train_and_ft = adata_test.X - means_train_and_ft
z_test_train_and_ft = z_test_train_and_ft @ PCs_train_and_ft
X_recon_test_train_and_ft = z_test_train_and_ft @ PCs_train_and_ft.T + means_train_and_ft

In [ ]:
adata_test_with_recon_train_and_ft = sc.AnnData(X=np.concatenate([adata_test.X.toarray(), X_recon_test_train_and_ft], axis=0))
obs_test_with_recon_train_and_ft = pd.DataFrame({"dataset_type": ["true"] * adata_test.n_obs + ["recon"] * adata_test.n_obs})
adata_test_with_recon_train_and_ft.obs = obs_test_with_recon_train_and_ft
sc.pp.pca(adata_test_with_recon_train_and_ft)
sc.pp.neighbors(adata_test_with_recon_train_and_ft)
sc.tl.umap(adata_test_with_recon_train_and_ft)
sc.pl.umap(adata_test_with_recon_train_and_ft, color=["dataset_type"], s=5)